# 03 · Gold — um dado, dois consumos

```
silver ─► gold.vendas_diarias ─┐
silver ─► gold.clientes_360    ├─► Iceberg ─► Postgres (schema gold) ─► Metabase   [BI]
silver ─► gold.avaliacoes_resumo┘

silver.avaliacoes ─► gold.avaliacao_chunks ─► Iceberg ─► pgvector (schema vetorial) [próximo módulo]
```

A Gold tem **dois públicos**: pessoas (painéis) e, em seguida, um modelo de linguagem (busca por significado).
Em ambos, o **Iceberg continua sendo a fonte da verdade**; o Postgres é uma *cópia para servir*.

In [9]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))          # para importar a pasta common/
from common import lake_local as L, medalhao as M
from pyspark.sql import functions as F
spark = L.get_spark()                              # PySpark local: sem cluster
print("Spark", spark.version, "| master:", spark.sparkContext.master, "| storage:", L.STORAGE)

Spark 3.5.3 | master: local[2] | storage: minio


## Parte A · Gold para BI

In [10]:
vd  = M.build_vendas_diarias(spark)
c360 = M.build_clientes_360(spark)
ares = M.build_avaliacoes_resumo(spark)
for t in ("vendas_diarias","clientes_360","avaliacoes_resumo"):
    print(t, spark.table(f"lake.gold.{t}").count(), "linhas")
c360.groupBy("segmento").count().orderBy("count", ascending=False).show()

vendas_diarias 30129 linhas
clientes_360 5000 linhas
avaliacoes_resumo 112 linhas
+----------+-----+
|  segmento|count|
+----------+-----+
|     ativo| 1713|
|   campeao| 1656|
|  em_risco|  640|
|sem_compra|  515|
|   inativo|  476|
+----------+-----+



### Servir: Gold (Iceberg) → Postgres
O Metabase não lê Iceberg. Ele lê o Postgres. `truncate=true` reaproveita a tabela, então os painéis não quebram a cada execução.

In [11]:
# Garante que o endereço aponta para a porta nova 5433
L.PG["port"] = 5433
print("Conectando em:", L.PG)

Conectando em: {'host': 'localhost', 'port': 5433, 'user': 'aula', 'password': 'aula123', 'dbname': 'serving'}


In [13]:
# Cria o schema 'gold' no Postgres caso não exista
with L.pg_conn() as conn:
    with conn.cursor() as cur:
        cur.execute("CREATE SCHEMA IF NOT EXISTS gold;")
    conn.commit()
print("Schema gold pronto!")

Schema gold pronto!


In [14]:
for nome, df in (("vendas_diarias", vd), ("clientes_360", c360), ("avaliacoes_resumo", ares)):
    print("publicado:", L.publish_to_postgres(df, nome, schema="gold"))
L.pg_query("SELECT categoria, round(sum(receita)) AS receita FROM gold.vendas_diarias GROUP BY 1 ORDER BY 2 DESC LIMIT 5")

publicado: gold.vendas_diarias
publicado: gold.clientes_360
publicado: gold.avaliacoes_resumo


c:\Users\giova\Downloads\arqu_coleta_dados\common\lake_local.py:148: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, c, params=params)


,categoria,receita
0,Alimentos,8753116.0
1,Eletrônicos,8380779.0
2,Moda,7896713.0
3,Esporte,7786758.0
4,Casa,7493598.0


### Metabase (BI básico)
1. Abra <http://localhost:3000> e crie o usuário de administração.
2. *Adicionar banco* → **PostgreSQL** → host `postgres` (é o nome do serviço dentro do Docker), porta `5432`, banco `serving`, usuário `aula`, senha `aula123`.
3. *Sincronizar esquema* e explore o schema **gold**. Monte 3 perguntas:
   - **Receita por mês** (gráfico de linha): `vendas_diarias`, somar `receita` por mês de `data`.
   - **Clientes por segmento** (barras): `clientes_360`, contar por `segmento`.
   - **Nota média por categoria** (barras): `avaliacoes_resumo`.
4. Junte as três num *dashboard*.